# BioEmu Colab-to-Graph Workflow

This notebook walks through the full BioEmu workflow for this project:

1. Prepare endpoint structures and a FASTA sequence locally.
2. Run BioEmu sampling in Google Colab using `run_bioemu_colab.ipynb`.
3. Download or sync the BioEmu output bundle.
4. Load the BioEmu samples as graph nodes.
5. Build the A-to-B conformational graph and find the shortest path.
6. Inspect graph, path, py3Dmol, PyMOL, and morph-GIF outputs.

Scientific caveat: BioEmu-generated conformations are treated as candidate graph nodes. The shortest path is a proposed structural path, not a validated physical transition pathway.

## 1. Data Locations

Put files in these locations before running the local graph analysis cells:

```text
data/bioemu_inputs/sequence.fasta      # FASTA used by BioEmu in Colab
data/pdb_inputs/A.pdb                  # endpoint A C-alpha-compatible PDB
data/pdb_inputs/B.pdb                  # endpoint B C-alpha-compatible PDB
data/bioemu_outputs/latest/            # downloaded BioEmu bundle
  manifest.json
  nodes/*.pdb
  raw_bioemu/                          # optional raw BioEmu outputs
```

The Colab notebook writes the BioEmu bundle. This notebook consumes that bundle.

In [ ]:
from pathlib import Path
import json

from IPython.display import Image, display

from guided_conformational_path.bioemu_experiment import (
    BioEmuPrecomputedConfig,
    run_bioemu_precomputed_experiment,
)
from guided_conformational_path.bioemu_io import load_bioemu_manifest, read_fasta
from guided_conformational_path.large_synthetic import (
    display_py3dmol_path,
    format_run_summary,
    write_run_outputs,
)
from guided_conformational_path.visualization import plot_graph

## 2. Configure Input Paths

Edit these paths if you use a different folder. The defaults match the repository data layout.

In [ ]:
sequence_fasta = Path("data/bioemu_inputs/sequence.fasta")
endpoint_a_file = Path("data/pdb_inputs/A.pdb")
endpoint_b_file = Path("data/pdb_inputs/B.pdb")
bioemu_bundle_dir = Path("data/bioemu_outputs/latest")
outputs_dir = Path("outputs/bioemu_experiment")

paths = {
    "sequence_fasta": sequence_fasta,
    "endpoint_a_file": endpoint_a_file,
    "endpoint_b_file": endpoint_b_file,
    "bioemu_bundle_dir": bioemu_bundle_dir,
    "bioemu_manifest": bioemu_bundle_dir / "manifest.json",
    "outputs_dir": outputs_dir,
}

for label, path in paths.items():
    print(f"{label:18s} {path} | exists={path.exists()}")

## 3. Colab Sampling Step

BioEmu itself is Linux/CUDA-oriented and is not expected to run natively on macOS.

Use this notebook in Colab to generate samples:

```text
notebooks/run_bioemu_colab.ipynb
```

In Colab, set:

```python
FASTA_PATH = Path("/content/drive/MyDrive/bioemu_inputs/protein.fasta")
OUTPUT_DIR = Path("/content/drive/MyDrive/bioemu_outputs/protein_bundle")
N_SAMPLES = 200
```

After the Colab run completes, copy or sync `OUTPUT_DIR` to your local machine as `data/bioemu_outputs/latest/`.

## 4. Inspect FASTA And BioEmu Bundle

These cells verify that the local files are readable before running graph analysis.

In [ ]:
if sequence_fasta.exists():
    sequence_id, sequence = read_fasta(sequence_fasta)
    print("Sequence ID:", sequence_id)
    print("Sequence length:", len(sequence))
else:
    print(f"Missing FASTA: {sequence_fasta}")
    print("This is only needed for recordkeeping locally; BioEmu sampling happens in Colab.")

In [ ]:
if (bioemu_bundle_dir / "manifest.json").exists():
    manifest = load_bioemu_manifest(bioemu_bundle_dir)
    print("Bundle schema:", manifest.get("schema_version"))
    print("Sequence:", manifest.get("sequence", {}).get("id"))
    print("Requested samples:", manifest.get("n_samples_requested"))
    print("Available samples:", len(manifest.get("samples", [])))
else:
    manifest = None
    print(f"Missing BioEmu manifest: {bioemu_bundle_dir / 'manifest.json'}")
    print("Run the Colab notebook first, then copy the output bundle here.")

## 5. Configure The Graph Experiment

`max_bioemu_samples=200` matches the current default BioEmu sample count. Lower it if the graph is too dense or slow.

In [ ]:
config = BioEmuPrecomputedConfig(
    endpoint_a_path=endpoint_a_file,
    endpoint_b_path=endpoint_b_file,
    bioemu_bundle_dir=bioemu_bundle_dir,
    max_bioemu_samples=200,
    k_neighbors=0,
    outputs_dir=outputs_dir,
    write_nodes=False,
    write_pymol=True,
    write_py3dmol=True,
    write_animation=True,
    animation_frames_per_edge=20,
    animation_fps=10,
    animation_dpi=120,
    graph_min_edge_weight=None,
    graph_max_edge_weight=None,
    graph_non_path_edge_alpha=0.125,
    matching_method="exact_then_geometric",
    min_exact_match_fraction=0.8,
    geometric_match_distance_cutoff=5.0,
    min_geometric_matches=8,
)
config

## 6. Run The BioEmu Graph Experiment

This loads A/B endpoint PDBs, loads BioEmu generated structures as neutral `external` nodes, builds a weighted graph, and finds the shortest A-to-B path.

In [ ]:
run = run_bioemu_precomputed_experiment(config)
write_run_outputs(run, config)
print(format_run_summary(run, config.outputs_dir))

## 7. Inspect Summary And Path

In [ ]:
run.summary

In [ ]:
{
    "path_node_ids": run.path_result.node_ids,
    "total_cost": run.path_result.total_cost,
    "path_intermediates": run.summary.get("path_intermediates"),
    "loaded_bioemu_nodes": run.summary.get("n_generated_nodes"),
}

## 8. Visualize Outputs

The graph PNG shows the abstract weighted graph. The GIF morphs the selected best path. The py3Dmol view shows the selected path conformations.

In [ ]:
graph_path = config.outputs_dir / "graph.png"
if graph_path.exists():
    display(Image(filename=graph_path))
else:
    print("Missing graph image:", graph_path)

In [ ]:
gif_path = run.summary.get("path_morph_gif")
if gif_path:
    display(Image(filename=gif_path))
else:
    print("No path morph GIF was written. Check config.write_animation.")

In [ ]:
display_py3dmol_path(run)

## 9. Output File Reference

In [ ]:
{
    "experiment_summary": config.outputs_dir / "experiment_summary.json",
    "path_summary": config.outputs_dir / "path_summary.json",
    "graph_png": config.outputs_dir / "graph.png",
    "path_morph_gif": config.outputs_dir / "path_morph.gif",
    "py3dmol_html": config.outputs_dir / "py3dmol_path_view.html",
    "pymol_script": config.outputs_dir / "pymol_path" / "view_path.pml",
}